# 💾 HiSparse — DeepSeek-V4 的分层显存架构

**本文目标**：深入理解 HiSparse 如何通过 CPU offloading 将长上下文推理的容量提升 3 倍。

读完这篇你会理解：
- 为什么 C4 层是 offloading 的最佳目标
- HiSparse Coordinator 的 LRU 换页机制
- GPU↔CPU 数据流和异步传输
- 与 vLLM swap 的差异 (HiSparse 是"常态", swap 是"应急")

## 1. 为什么只卸载 C4？

### 1.1 三个池的访问模式分析

```
SWA 池 (128 tokens):
  - 大小: 很小 (128 × per_token_size)
  - 访问: 每一步每一个 token 都访问 (因为总是看最近 128)
  - → 100% 活跃 → 不适合卸载

C128 池 (全量密集):
  - 大小: 中等 (seq_len / 128)
  - 访问: 每一步访问全部 (所有压缩位置)
  - → 100% 活跃 → 不适合卸载

C4 池 (sparse top-512):
  - 大小: 很大 (seq_len / 4 = 250K for 1M)
  - 访问: 每一步只访问 top-512 (由 indexer 选择)
  - → 0.2% 活跃! (512 / 250,000)
  - → 绝大多数 C4 KV 在任一步都是 "闲置" 的 → 适合卸载!
```

### 1.2 核心洞察

```
"Most C4 KV is inactive at any moment and can live on CPU"

  250K C4 压缩 tokens
  - 其中只有 512 个在当前的 attention step 被访问
  - 其余 249,488 个只是 "存在但不被使用"
  
  → 把它们放在 CPU 内存
  → 当它们被 indexer 的 top-k 选中时 → 异步换入 GPU
  → 当前未被选中的 → 放在 CPU (不占 GPU 显存)
```

## 2. HiSparse 架构

### 2.1 整体设计

```
┌─────────────────────────────────────────────────────────────┐
│                    HiSparse 分层显存                          │
├─────────────────────────────────────────────────────────────┤
│                                                               │
│  GPU HBM (高速, 小容量):                                       │
│  ┌──────────────────────────────────────────────────┐        │
│  │  Model Weights (FP4/FP8)  │  SWA KV (小, 固定)    │        │
│  │  C128 KV (中等, 全保留)   │  Active C4 KV (精选)   │        │
│  │                          │  (当前 top-512 和相关)  │        │
│  └──────────────────────────────────────────────────┘        │
│                          │                                    │
│                     PCIe (~50 GB/s)                            │
│                          │                                    │
│  CPU DRAM (低速, 大容量):                                      │
│  ┌──────────────────────────────────────────────────┐        │
│  │              C4 KV Cache Pool                      │        │
│  │  (完整的上下文, 所有 C4 压缩位置)                    │        │
│  │                                                    │        │
│  │  使用 CPU pinned memory (加速 GPU↔CPU 传输)       │        │
│  │  通过 HiSparse Coordinator 管理                   │        │
│  └──────────────────────────────────────────────────┘        │
│                                                               │
└─────────────────────────────────────────────────────────────┘
```

### 2.2 HiSparse Coordinator

```python
class HiSparseCoordinator:
    """管理 C4 KV 的 GPU↔CPU 迁移"""
    
    def __init__(self):
        self.gpu_buffer = LRUCache(capacity=ACTIVE_PAGES)  # GPU 活跃页
        self.cpu_pool = CPUPinnedMemory()                  # CPU pinned 内存
    
    def on_decode_step(self, c4_topk_indices):
        """每个 decode step 调用"""
        needed_pages = self.indices_to_pages(c4_topk_indices)
        
        # 1. 检查哪些页在 GPU → CPU
        gpu_hits = [p for p in needed_pages if p in self.gpu_buffer]
        cpu_needed = [p for p in needed_pages if p not in self.gpu_buffer]
        
        # 2. 从 GPU buffer 淘汰最久未使用的页
        while not self.gpu_buffer.has_space(len(cpu_needed)):
            victim = self.gpu_buffer.evict_lru()
            # 如果页是 dirty (被更新过) → 异步写回 CPU
            if victim.is_dirty:
                self.cpu_pool.async_write(victim)
        
        # 3. 从 CPU 换入需要的页
        for page in cpu_needed:
            data = self.cpu_pool.read(page)
            self.gpu_buffer.insert(page, data)
        
        # 4. KV 生成后 → 异步备份新生成的 C4 KV 到 CPU
        new_c4_kv = self.generate_c4_kv()
        self.cpu_pool.async_write_backup(new_c4_kv)
    
    def indices_to_pages(self, topk_indices):
        """将 C4 索引映射到内存页"""
        # C4 KV 以 page (如 64 个压缩位置) 为单位管理
        return set(idx // PAGE_SIZE for idx in topk_indices)
```

### 2.3 异步传输与 GPU 计算重叠

```
HiSparse 的关键性能优化: 异步传输

时间线 (单个 decode step):
  
  无 HiSparse:
    等待 GPU 空闲 → CPU→GPU 传输 → GPU 计算 → 等待
                    [~~~~~~~~50us~~~~~~~]
  
  有 HiSparse (overlap):
    GPU 计算 step N      [████████████████████]
    CPU→GPU 传输 step N+1     [~~~~~~~~]
    GPU→CPU 备份 step N-1           [~~~~~~~~]
    
    → 传输与计算重叠 → 几乎无额外延迟!
    
关键: 使用 CUDA stream 分离传输和计算流
  stream_compute: 运行 attention kernel
  stream_h2d:     CPU→GPU 传输 (cudaMemcpyAsync)
  stream_d2h:     GPU→CPU 备份 (cudaMemcpyAsync)
```

## 3. 性能与限制

### 3.1 收益量化

```
场景: DeepSeek-V4 Flash, 2xB200, 200K input / 20K output

无 HiSparse:
  C4 KV 全放 GPU → ~50 GB
  → GPU 显存限制 → 只能服务 N 个并发请求

有 HiSparse:
  C4 KV 90% 在 CPU → GPU 只保留 ~5 GB 活跃部分
  → GPU 显存释放 45 GB → 可服务 ~3x 并发请求
  → 或支持 3x 长的上下文

吞吐提升: 最高 3x (具体取决于 batch size 和上下文长度)
```

### 3.2 适用条件

```
HiSparse 最有效的条件:
  1. C4 层占多数 (V4 满足)
  2. TopK 选择稀疏 (K << 压缩总数, V4 是 512 << 250K, 满足)
  3. PCIe 带宽足够 (B200: 50 GB/s, 满足)
  4. 传输能与计算重叠 (需要精心设计的 CUDA stream)

HiSparse 不适合:
  - C128 层 (全量访问 → 100% 活跃 → 卸载无意义)
  - SWA (太小 → 卸载 overhead > 收益)
  - 短上下文 (C4 总数小 → 卸载收益有限)
```

### 3.4 与 vLLM Preemption Swap 的差异

| | vLLM Swap (抢占) | HiSparse |
|---|---|---|
| 触发条件 | 显存不足时 (应急) | **始终运行** (常态) |
| 交换对象 | 被抢占请求的全部 KV Cache | **C4 层的不活跃页** |
| 数据量 | 可能很大 (整个请求) | 小且可控 (按页) |
| 延迟影响 | 明显 (swap out/in 是同步阻塞) | **几乎无** (异步 + overlap) |
| 设计目标 | 防止 OOM | **扩展容量** |